# ERK-KTR whole-FOV stimulation on the Ti2-E (RTMSequence)

A multi-phase ERK-KTR optogenetic experiment with whole-FOV blue
light, adapted from the Pertzlab Jungfrau notebook.

**Workflow:**
1. Connect to the Ti2-E and define the channels (imaging, stimulation, optocheck)
2. Set up the pipeline (segmentation, tracking, ERK-KTR features)
3. Pick FOV positions in napari and build the phases with `RTMSequence`
4. Validate and run
5. Merge the tracks into one `exp_data.parquet`

**Channels:**

| Role | Jungfrau preset | Ti2-E preset |
|------|-----------------|--------------|
| Nuclear marker (H2B-miRFP), segmented | `miRFP` | `iRFP` |
| ERK-KTR reporter (mScarlet3) | `mScarlet3` | `RFP` |
| Stimulation, blue light, whole FOV | `CyanStim` | `GFP` |
| Optocheck (mCitrine) | `mCitrine` | `YFP_Custom` |

The Ti2-E has no DMD, so stimulation lights the whole field of view with the
blue excitation of the ET-GFP cube. The optocheck channel is excited by the
same cube, so it also stimulates; that is why it runs on the last frame only.
All presets share the X-Cite XT600 LED; the `power` of a `PowerChannel` sets
its intensity in percent. Exposures and powers are the Jungfrau values:
tune them for the XT600 before a real run.

In [ ]:
import os
import time

from faro.core.data_structures import (
    Channel,  # imaging channel (config, exposure, group)
    PowerChannel,  # channel with light-source power (adds power 0-100)
    RTMSequence,  # one phase: time plan, channels, stim and ref frames
    SegmentationMethod,
    combine,  # join phases along an axis (t or p)
)
import faro.core.utils as utils

### Microscope

`Ti2E()` loads `NikonTi2e.cfg`, applies `System/Startup` and selects the
`Channel` group, so channels need no `group`. Close NIS-Elements first.

In [ ]:
from faro_aoki.ti2e import Ti2E

mic = Ti2E()
utils.print_configs(mic.mmc)

### Experimental settings

- `SLEEP_BEFORE_EXPERIMENT_START_in_H`: delay before the first frame, in hours.
- `base_path` / `experiment_name`: output folder for the OME-Zarr store (or TIFFs), tracks and `exp_data.parquet`.
- `WRITER`: `"omezarr"` (one store) or `"tiff"` (one file per frame).
- `stim_channel`: the stimulation light; `power` is the LED intensity (0-100), `exposure` the pulse length in ms.
- `group=mic.CHANNEL_GROUP`: the `Channel` group; RTMSequence needs the group of a `PowerChannel`.
- `imaging_channels`: acquired on every frame. Channel 0 is segmented; `FE_ErkKtr` reads the reporter from channel 1.
- `optocheck_channel`: reference channel on the last frame only, to check the expression of the optogenetic tool.

In [ ]:
## Configuration
SLEEP_BEFORE_EXPERIMENT_START_in_H = 0  # 0 = start immediately

## Storage
base_path = os.path.join(os.path.expanduser("~"), "faro_data")  # TODO: data folder
experiment_name = "test"
path = os.path.join(base_path, experiment_name)
WRITER = "omezarr"  # or "tiff"

## Stimulation channel: blue light through the ET-GFP cube
stim_channel = PowerChannel(
    config="GFP", exposure=100, group=mic.CHANNEL_GROUP, power=10
)

## Imaging channels, on every frame (channel 0 is segmented)
imaging_channels = (
    PowerChannel(
        config="iRFP", exposure=100, group=mic.CHANNEL_GROUP, power=99
    ),  # H2B-miRFP
    PowerChannel(
        config="RFP", exposure=100, group=mic.CHANNEL_GROUP, power=99
    ),  # ERK-KTR-mScarlet3
)

## Optocheck: expression of the optogenetic tool (mCitrine), last frame only
optocheck_channel = PowerChannel(
    config="YFP_Custom", exposure=600, group=mic.CHANNEL_GROUP, power=99
)

### Pipeline

- **Segmentation**: Cellpose v4 on the nuclear marker.
- **Stimulator**: `StimWholeFOV()` lights the whole field of view.
- **Features**: `FE_ErkKtr("labels")` measures the reporter in the nucleus and in a cytoplasmic ring; `cnr` is the cytoplasm/nucleus ratio.
- **Tracker**: `TrackerTrackpy(search_range=50)`, at most 50 px between frames.
- **Optocheck**: `RefFE("labels")` measures the reference channel per cell on the reference frames.

In [ ]:
from faro.core.controller import Controller
from faro.core.pipeline import ImageProcessingPipeline
from faro.core.writers import OmeZarrWriter, TiffWriter
from faro.feature_extraction.erk_ktr import FE_ErkKtr
from faro.feature_extraction.ref import RefFE
from faro.segmentation.cellpose_v4 import CellposeV4
from faro.stimulation.base import StimWholeFOV
from faro.tracking.trackpy import TrackerTrackpy

# Cellpose needs `uv sync --extra cellpose`. Without it, OtsuSegmentator works
# for a bright nuclear marker:
# from faro.segmentation.base import OtsuSegmentator;  CellposeV4() -> OtsuSegmentator()

segmentators = [
    SegmentationMethod(
        name="labels",
        segmentation_class=CellposeV4(),
        use_channel=0,  # iRFP
        save_tracked=True,
    )
]

stimulator = StimWholeFOV()
feature_extractor = FE_ErkKtr("labels")
tracker = TrackerTrackpy(search_range=50)
optocheck = RefFE(used_mask="labels")

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    tracker=tracker,
    stimulator=stimulator,
    feature_extractor_ref=optocheck,  # runs on the ref (optocheck) frames
)

if WRITER == "omezarr":
    writer = OmeZarrWriter(storage_path=path)  # images + stim readout
else:
    writer = TiffWriter(storage_path=path)

### GUI: napari and Micro-Manager

Opens napari with the napari-micromanager widget. Use it for live view and
to pick the FOV positions in the MDA widget; the next steps read them from
there.

In [ ]:
import napari
from napari_micromanager import MainWindow

viewer = napari.Viewer()
mm_wdg = MainWindow(viewer, mmcore=mic.mmc)
viewer.window.add_dock_widget(mm_wdg)

### Build the acquisition events

`generate_fov_positions(mic, viewer=viewer)` reads the positions of the
napari MDA widget. Without the GUI, list them by hand with
`generate_fov_positions_from_list`.

`RTMSequence` parameters:
- `time_plan`: `{"interval": seconds, "loops": N}`
- `stage_positions`: the FOV positions
- `channels`: imaging channels on every frame
- `stim_channels` / `stim_frames`: stimulation on these frames of the phase (0-indexed, `-1` is the last frame)
- `stim_exposure`: one value or one per stim frame for ramps; `None` keeps the channel exposure
- `ref_channels` / `ref_frames`: reference (optocheck) channels on these frames
- `rtm_metadata`: travels with every event into the tracks

`combine(phase_1, phase_2, axis="t")` runs the phases one after the
other; frame indices are resolved per phase before joining.
`apply_fov_batching` splits the FOVs into batches when they do not
all fit into one interval.

In [ ]:
fov_positions = utils.generate_fov_positions(mic, viewer=viewer)
# Without the GUI:
# x0, y0 = mic.mmc.getXYPosition()
# fov_positions = utils.generate_fov_positions_from_list(mic, [{"x": x0, "y": y0}])

# Phase 1: baseline with one stim pulse
phase_1 = RTMSequence(
    time_plan={"interval": 10.0, "loops": 4},
    stage_positions=fov_positions,
    channels=imaging_channels,
    stim_channels=(stim_channel,),
    stim_frames=range(1, 2),  # frame 1 of this phase
    rtm_metadata={
        "phase_name": "PreDrug",
        "phase_id": 0,
        "treatment_name": "Priming Phase 1 pre Drug",
    },
)

# Phase 2: sustained stimulation, optocheck on the last frame
phase_2 = RTMSequence(
    time_plan={"interval": 10.0, "loops": 4},
    stage_positions=fov_positions,
    channels=imaging_channels,
    stim_channels=(stim_channel,),
    stim_frames=range(1, 3),  # frames 1 and 2 of this phase
    ref_channels=(optocheck_channel,),
    ref_frames=[-1],
    rtm_metadata={
        "phase_name": "PostDrug",
        "phase_id": 1,
        "treatment_name": "Sustained Phase 2 post Drug",
    },
)

events = combine(phase_1, phase_2, axis="t")
events = utils.apply_fov_batching(events, time_per_fov=2.0)

print(f"Total events: {len(events)}")
utils.events_to_dataframe(events).sort_values("timestep")

### Validate the experiment

`validate_events` checks the pipeline components, the event metadata,
the channel names, the exposures and the LED power range.

`stim_mode` of `run_experiment`:
- `"current"`: acquire, run the pipeline, then stimulate in the same timepoint
- `"previous"`: stimulate with the mask of the previous timepoint (lower latency)

In [ ]:
ctrl = Controller(mic, pipeline, writer=writer)
assert ctrl.validate_events(events), "fix the warnings above before running"

In [ ]:
# Live status with Start/Stop; re-binds when a run starts.
from faro.widgets import ExperimentStatusWidget

viewer.window.add_dock_widget(
    ExperimentStatusWidget(ctrl), name="experiment status", area="right"
)

### Run the experiment

`run_experiment` returns at once; the acquisition and the pipeline run
in the background, and napari keeps showing the frames.

In [ ]:
for _ in range(int(SLEEP_BEFORE_EXPERIMENT_START_in_H * 3600)):
    time.sleep(1)

handle = ctrl.run_experiment(events, stim_mode="current")

### Post-processing

`finish_experiment()` waits for the run, drains the pipeline and closes
the writer. `generate_exp_data_from_tracks` merges the per-FOV tracks
into `exp_data.parquet`.

In [ ]:
handle.wait()
ctrl.finish_experiment()
mic.post_experiment()
utils.generate_exp_data_from_tracks(path)

In [ ]:
import pandas as pd

exp_data = pd.read_parquet(os.path.join(path, "exp_data.parquet"))
n_tracks = exp_data.groupby(["fov", "particle"]).ngroups
print(f"{n_tracks} tracks in {exp_data['fov'].nunique()} FOVs")
ax = exp_data.groupby("timestep")["cnr"].mean().plot(marker="o")
ax.set_xlabel("timestep")
ax.set_ylabel("mean ERK-KTR C/N ratio")